# Iron March — 03: Embeddings y Perfilado
Representación vectorial de usuarios, reducción dimensional, estilometría y atribución.
Estrategias: embed_users (A), centroids full (B), centroids sampled top-50/100/150 (C/D/E).

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import re
from collections import Counter

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import umap as umap_lib
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

from src.utils import load_or_compute, RESULTS_DIR, DATA_DIR
from src.embeddings import embed_users, compute_actor_centroids
from src.stylometry import compare_users, extract_features

plt.style.use('dark_background')
sns.set_palette('muted')

IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Datos:      {DATA_DIR}")
print(f"Resultados: {IM_RESULTS}")

In [ ]:
# ── Load pre-processed parquets (output of notebooks 01/02) ──────────────────
posts_clean = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')
users_clean = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')

print(f"Posts:    {len(posts_clean):,}")
print(f"Usuarios: {len(users_clean):,}")
print(f"Cols posts: {list(posts_clean.columns)}")
print(f"Cols users: {list(users_clean.columns)}")

# ── Convenience maps ─────────────────────────────────────────────────────────
uid_to_name = dict(zip(users_clean['userid'], users_clean['username']))

# ── Helper: load most recent .npz matching glob pattern ──────────────────────
def load_latest(pattern: str) -> dict | None:
    """Return dict from most-recent .npz in IM_RESULTS matching pattern, or None."""
    matches = sorted(IM_RESULTS.glob(pattern))
    if not matches:
        return None
    path = matches[-1]
    print(f"Cargando: {path.name}")
    return dict(np.load(path, allow_pickle=True))

def save_timestamped(data: dict, section: str, name: str) -> Path:
    """Save a timestamped .npz to IM_RESULTS."""
    ts  = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = IM_RESULTS / f"{section}_{name}_{ts}.npz"
    np.savez_compressed(out, **data)
    print(f"Guardado: {out.name}")
    return out

print("Helpers listos.")

## Sección 1 — Experimento metodológico: embeddings

Tres enfoques distintos para representar usuarios como vectores.
El objetivo es entender qué se pierde (o no) al muestrear.

| Parte | Método | Posts procesados | Tiempo real | Spearman ρ vs B | Guardado |
|---|---|---|---|---|---|
| A | `embed_users` — concatenar todos los posts → 1 vector | 157,779 | ~12 min | — | `s5a_embed_users_*.npz` |
| B | `compute_actor_centroids` **completo** — 1 embedding por post, promedio | 157,779 | ~9 h | ref. | `s5b_centroids_full_*.npz` |
| C | `compute_actor_centroids` **top-50** posts más largos/usuario | ≤41,800 | **2h 35min** | **ρ=0.799** | `s5c_centroids_sampled_*.npz` |
| D | Idem **top-100** posts más largos/usuario | 41,319 | **3h 33min** | **ρ=0.860** | `s5d_centroids_sampled100_*.npz` |
| E | Idem **top-150** posts más largos/usuario | 53,193 | **4h 12min** | **ρ=0.896** | `s5e_centroids_sampled150_*.npz` |

> Todos los tiempos medidos con `qwen3-embedding` (4096D) sobre GPU local.
> El salto de C→D (+37min, +6 puntos ρ) es más eficiente que D→E (+39min, +4 puntos ρ).
> Para análisis exploratorio, **D es el mejor balance coste/fidelidad**.

Las celdas cargan el resultado más reciente si existe; si no, lo computan y guardan.

In [ ]:
# Prepare posts DataFrame for embedding functions
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'

posts_embed = posts_clean[['userid', text_col]].copy()
posts_embed = posts_embed.dropna(subset=[text_col])
posts_embed = posts_embed[posts_embed[text_col].astype(str).str.strip().str.len() > 20]
posts_embed = posts_embed.rename(columns={text_col: 'pagetext'})

post_counts = posts_embed.groupby('userid').size()
print(f"Posts con texto válido: {len(posts_embed):,}")
print(f"Usuarios con ≥5 posts:  {(post_counts >= 5).sum():,}")
print(f"Usuarios con ≥10 posts: {(post_counts >= 10).sum():,}")

In [ ]:
# Part A — embed_users: concatenate all posts per user → 1 vector per user
# Estimated: ~12 min  |  Cache: s5a_embed_users_*.npz
#
# MAX_CHARS truncates the concatenated text per user before embedding.
# Without it, very prolific users (e.g. Slavros ~5.7M chars) block Ollama indefinitely.
MAX_CHARS_A = 50_000

cached_a = load_latest('s5a_embed_users_*.npz')

if cached_a is not None:
    user_ids_a = cached_a['user_ids'].tolist()
    vectors_a  = cached_a['vectors']
    print(f"Parte A cargada: {len(user_ids_a):,} usuarios, dim={vectors_a.shape[1]}")
else:
    import ollama as _ollama
    from tqdm import tqdm as _tqdm

    def _embed_users_truncated(posts_df, text_col='pagetext', min_posts=5):
        df = posts_df.dropna(subset=['userid', text_col]).copy()
        df[text_col] = df[text_col].astype(str).str.strip()
        df = df[df[text_col].str.len() > 0]
        post_counts = df.groupby('userid').size()
        df = df[df['userid'].isin(post_counts[post_counts >= min_posts].index)]
        grouped = df.groupby('userid')[text_col].apply(lambda x: ' '.join(x.tolist()))
        uids = grouped.index.tolist()
        texts = [t[:MAX_CHARS_A] for t in grouped.tolist()]
        print(f"  Usuarios: {len(uids):,} | texto max tras truncar: {max(len(t) for t in texts):,} chars")
        results = []
        batches = [texts[i:i+32] for i in range(0, len(texts), 32)]
        for batch in _tqdm(batches, desc='Embedding [qwen3-embedding]'):
            resp = _ollama.embed(model='qwen3-embedding', input=batch)
            results.extend(resp.embeddings)
        return uids, np.array(results, dtype=np.float32)

    print("Computando Parte A (embed_users con truncado a 50K chars)...")
    user_ids_a, vectors_a = _embed_users_truncated(posts_embed)
    save_timestamped(
        {'user_ids': np.array(user_ids_a, dtype='U64'), 'vectors': vectors_a},
        's5a', 'embed_users',
    )
    print(f"Parte A completada: {len(user_ids_a):,} usuarios")

In [ ]:
# Part B — compute_actor_centroids (all posts): 1 embedding per post, mean per user
# ⚠️ Estimated: 6–10 h  |  Cache: s5b_centroids_full_*.npz
cached_b = load_latest('s5b_centroids_full_*.npz')

if cached_b is not None:
    user_ids_b = cached_b['user_ids'].tolist()
    vectors_b  = cached_b['vectors']
    print(f"Parte B cargada: {len(user_ids_b):,} usuarios, dim={vectors_b.shape[1]}")
else:
    print(f"Computando Parte B — todos los posts ({len(posts_embed):,})...")
    print("Estimado: 6–10 h. Dejar corriendo.")
    user_ids_b, vectors_b = compute_actor_centroids(posts_embed, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(user_ids_b, dtype='U64'), 'vectors': vectors_b},
        's5b', 'centroids_full',
    )
    print(f"Parte B completada: {len(user_ids_b):,} usuarios")

In [ ]:
# Parts C / D / E — stratified sample: top-N longest posts per user
# C → top-50   (~1.5 h)  s5c_centroids_sampled_*.npz
# D → top-100  (~3 h)    s5d_centroids_sampled100_*.npz
# E → top-150  (~4 h)    s5e_centroids_sampled150_*.npz

def build_sampled_centroids(max_posts: int, section: str, cache_pattern: str):
    cached = load_latest(cache_pattern)
    if cached is not None:
        ids  = cached['user_ids'].tolist()
        vecs = cached['vectors']
        print(f"Cargado (top-{max_posts}): {len(ids):,} usuarios, dim={vecs.shape[1]}")
        return ids, vecs

    sample = (
        posts_embed
        .assign(text_len=posts_embed['pagetext'].str.len())
        .sort_values('text_len', ascending=False)
        .groupby('userid')
        .head(max_posts)
        .drop(columns='text_len')
        .reset_index(drop=True)
    )
    print(f"Computando (top-{max_posts}): {len(sample):,} posts...")
    ids, vecs = compute_actor_centroids(sample, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(ids, dtype='U64'), 'vectors': vecs},
        section, f'centroids_sampled{max_posts}',
    )
    print(f"Completado (top-{max_posts}): {len(ids):,} usuarios")
    return ids, vecs

user_ids_c, vectors_c = build_sampled_centroids(50,  's5c', 's5c_centroids_sampled_*.npz')
user_ids_d, vectors_d = build_sampled_centroids(100, 's5d', 's5d_centroids_sampled100_*.npz')
user_ids_e, vectors_e = build_sampled_centroids(150, 's5e', 's5e_centroids_sampled150_*.npz')

In [ ]:
# Spearman comparison: all methods vs full centroids (B)
# High ρ → sampling preserves the pairwise similarity ranking

def pairwise_sims(user_ids: list, vectors: np.ndarray) -> pd.Series:
    """Cosine similarities for all upper-triangle pairs."""
    sim_matrix = cosine_similarity(vectors)
    np.fill_diagonal(sim_matrix, 0)
    idx_i, idx_j = np.triu_indices(len(user_ids), k=1)
    keys = pd.MultiIndex.from_arrays([
        [str(user_ids[i]) for i in idx_i],
        [str(user_ids[j]) for j in idx_j],
    ])
    return pd.Series(sim_matrix[idx_i, idx_j], index=keys)

available = {}
for label, ids, vecs in [
    ('A: embed_users',        'user_ids_a', 'vectors_a'),
    ('B: centroids_full',     'user_ids_b', 'vectors_b'),
    ('C: sampled_top50',      'user_ids_c', 'vectors_c'),
    ('D: sampled_top100',     'user_ids_d', 'vectors_d'),
    ('E: sampled_top150',     'user_ids_e', 'vectors_e'),
]:
    try:
        _ids  = eval(ids)
        _vecs = eval(vecs)
        if len(_ids) > 1:
            print(f"Calculando pares para {label}...", flush=True)
            available[label] = pairwise_sims(_ids, _vecs)
    except NameError:
        pass

print("\nCorrelaciones de Spearman vs método B (centroids_full):")
spearman_results = {}
ref_label = 'B: centroids_full'
if ref_label in available:
    for lbl, sims in available.items():
        if lbl == ref_label:
            continue
        common = sims.index.intersection(available[ref_label].index)
        if len(common) == 0:
            continue
        rho, p = spearmanr(sims[common], available[ref_label][common])
        spearman_results[lbl] = rho
        print(f"  {lbl:<25} ρ={rho:.4f}  (p={p:.2e}, n={len(common):,})")
else:
    print("  Parte B no disponible; comparando entre métodos presentes.")
    keys = list(available.keys())
    for i in range(len(keys)):
        for j in range(i+1, len(keys)):
            lx, ly = keys[i], keys[j]
            common = available[lx].index.intersection(available[ly].index)
            if len(common) == 0:
                continue
            rho, p = spearmanr(available[lx][common], available[ly][common])
            spearman_results[(lx, ly)] = rho
            print(f"  {lx} vs {ly}: ρ={rho:.4f}")

In [ ]:
# UMAP comparative subplots — one panel per embedding method
method_data = {}
for label, ids_name, vecs_name in [
    ('A: embed_users',    'user_ids_a', 'vectors_a'),
    ('B: centroids_full', 'user_ids_b', 'vectors_b'),
    ('C: sampled_top50',  'user_ids_c', 'vectors_c'),
    ('D: sampled_top100', 'user_ids_d', 'vectors_d'),
    ('E: sampled_top150', 'user_ids_e', 'vectors_e'),
]:
    try:
        method_data[label] = (eval(ids_name), eval(vecs_name))
    except NameError:
        pass

if not method_data:
    print("Sin embeddings disponibles. Ejecutar primero las partes A–E.")
else:
    n_methods = len(method_data)
    fig = make_subplots(
        rows=1, cols=n_methods,
        subplot_titles=list(method_data.keys()),
        horizontal_spacing=0.04,
    )

    for col_idx, (label, (ids, vecs)) in enumerate(method_data.items(), 1):
        print(f"UMAP {label}...", end=' ', flush=True)
        reducer  = umap_lib.UMAP(n_components=2, n_neighbors=min(15, len(vecs) - 1), random_state=42)
        coords   = reducer.fit_transform(vecs)
        clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
        clusters  = clusterer.fit_predict(vecs)
        n_cl = len(set(clusters)) - (1 if -1 in clusters else 0)
        print(f"{n_cl} clusters, {(clusters == -1).sum()} ruido")
        names = [uid_to_name.get(uid, str(uid)) for uid in ids]

        fig.add_trace(
            go.Scatter(
                x=coords[:, 0], y=coords[:, 1],
                mode='markers',
                marker=dict(size=5, color=clusters, colorscale='Plasma', opacity=0.8,
                            line=dict(width=0)),
                text=[f'{n}<br>cluster={c}' for n, c in zip(names, clusters)],
                hoverinfo='text',
                showlegend=False,
            ),
            row=1, col=col_idx,
        )
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=col_idx)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=col_idx)

    fig.update_layout(
        title='UMAP + HDBSCAN — comparativa de métodos de embedding (hover = usuario)',
        template='plotly_dark',
        height=520, width=400 * n_methods,
        margin=dict(l=10, r=10, t=80, b=10),
    )
    fig.show()

## Sección 2 — Análisis de similitud y clusters

Resultados del experimento de muestreo (Spearman ρ vs centroides completos B):

| Estrategia | Posts/usuario | Tiempo | ρ vs full | Δρ vs anterior |
|---|---|---|---|---|
| C — top-50  | ≤41,800 | 2h 35min | 0.799 | — |
| D — top-100 | 41,319  | 3h 33min | 0.860 | +0.061 |
| E — top-150 | 53,193  | 4h 12min | 0.896 | +0.036 |

El rendimiento marginal decrece: cada 37–39 minutos adicionales compran menos ρ.
**D (top-100) es el punto óptimo** para este corpus: 86% de fidelidad al full con el 40% del tiempo.

Para el análisis de esta sección se prefiere el mejor embedding disponible (B > E > D > C > A).

In [ ]:
# Choose best available embedding (prefer B > E > D > C > A)
_priority = [
    ('B: centroids_full',  'user_ids_b', 'vectors_b'),
    ('E: sampled_top150',  'user_ids_e', 'vectors_e'),
    ('D: sampled_top100',  'user_ids_d', 'vectors_d'),
    ('C: sampled_top50',   'user_ids_c', 'vectors_c'),
    ('A: embed_users',     'user_ids_a', 'vectors_a'),
]

best_label, best_ids, best_vecs = None, None, None
for label, ids_name, vecs_name in _priority:
    try:
        _ids  = eval(ids_name)
        _vecs = eval(vecs_name)
        if len(_ids) > 1:
            best_label, best_ids, best_vecs = label, _ids, _vecs
            break
    except NameError:
        pass

if best_label:
    print(f"Método seleccionado: {best_label}")
    print(f"  Usuarios: {len(best_ids):,}  |  dim: {best_vecs.shape[1]}")
else:
    print("⚠️ Sin embeddings disponibles — ejecutar Sección 1 primero.")

In [ ]:
# UMAP 2D + HDBSCAN clusters — interactive Plotly scatter
if best_vecs is not None:
    print("Calculando UMAP...", flush=True)
    reducer   = umap_lib.UMAP(n_components=2, n_neighbors=min(15, len(best_ids) - 1), random_state=42)
    coords_2d = reducer.fit_transform(best_vecs)

    print("Calculando HDBSCAN...", flush=True)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
    clusters  = clusterer.fit_predict(best_vecs)
    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    print(f"Clusters encontrados: {n_clusters}  |  ruido: {(clusters == -1).sum()}")

    names = [str(uid_to_name.get(uid, uid)) for uid in best_ids]

    fig = go.Figure(go.Scatter(
        x=coords_2d[:, 0], y=coords_2d[:, 1],
        mode='markers',
        marker=dict(
            size=6,
            color=clusters,
            colorscale='Plasma',
            opacity=0.85,
            showscale=True,
            colorbar=dict(title='Cluster'),
            line=dict(width=0),
        ),
        text=[f'<b>{n}</b><br>cluster={c}' for n, c in zip(names, clusters)],
        hovertemplate='%{text}<extra></extra>',
    ))
    fig.update_layout(
        title=f'UMAP + HDBSCAN — {best_label} ({n_clusters} clusters)<br>'
              '<sup>Color = cluster | gris = ruido | hover = usuario</sup>',
        template='plotly_dark', height=620, width=900,
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    )
    fig.show()
else:
    print("Sin embeddings. Ejecutar Sección 1.")

In [ ]:
# Print members of each cluster (excluding noise cluster -1)
if best_vecs is not None and 'clusters' in dir():
    cluster_ids = sorted(set(clusters) - {-1})
    print(f"Total clusters: {len(cluster_ids)}")
    for cid in cluster_ids:
        members = [str(uid_to_name.get(uid, uid))
                   for uid, cl in zip(best_ids, clusters) if cl == cid]
        print(f"\nCluster {cid} ({len(members)} usuarios):")
        print("  " + ", ".join(members[:30]) + ("..." if len(members) > 30 else ""))

## Sección 3 — Estilometría: Burrows' Delta

**Burrows' Delta** es el estándar en atribución de autoría.
En lugar de features de superficie (puntuación, longitud), mide la frecuencia relativa de las
palabras más comunes del corpus, normalizadas por z-score.

La intuición: los autores usan palabras función (preposiciones, artículos, conjunciones) de
forma inconsciente y consistente, independientemente del tema.

**Delta bajo entre dos usuarios** → estilo similar → candidatos a misma persona con cuentas distintas (sockpuppet).

Requiere mínimo de texto por usuario para ser estadísticamente fiable.

In [ ]:
# Prepare corpus for Burrows' Delta
# Minimum 500 words per user — less text = unreliable Delta
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'
MIN_WORDS = 500

def clean_text(t: str) -> str:
    t = re.sub(r'<[^>]+>', ' ', str(t))   # strip HTML
    t = re.sub(r'https?://\S+', ' ', t)   # strip URLs
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)    # letters only
    return t.lower()

records = []
for uid, group in posts_clean.groupby('userid'):
    uname = uid_to_name.get(uid, uid)
    if pd.isna(uname):
        continue
    combined = ' '.join(group[text_col].dropna().astype(str).tolist())
    words    = clean_text(combined).split()
    if len(words) >= MIN_WORDS:
        records.append({'userid': uid, 'username': str(uname), 'words': words, 'n_words': len(words)})

delta_corpus = pd.DataFrame(records).sort_values('n_words', ascending=False).reset_index(drop=True)
print(f"Usuarios con ≥{MIN_WORDS} palabras: {len(delta_corpus):,}")
print(f"  Mediana palabras: {delta_corpus['n_words'].median():,.0f}")
print(f"\nTop 10:")
print(delta_corpus[['username', 'n_words']].head(10).to_string(index=False))

In [ ]:
# Compute Burrows' Delta — top-200 most frequent words, top-150 users
N_FEATURES = 200
TOP_USERS  = 150

# Corpus vocabulary
all_words = [w for row in delta_corpus['words'] for w in row]
vocab      = [w for w, _ in Counter(all_words).most_common(N_FEATURES)]
vocab_set  = set(vocab)
print(f"Top {N_FEATURES} function words: {', '.join(vocab[:30])} ...")

top_users = delta_corpus.head(TOP_USERS).copy()

# Relative frequency matrix
freq_matrix = np.zeros((len(top_users), N_FEATURES))
for i, (_, row) in enumerate(top_users.iterrows()):
    counts = Counter(w for w in row['words'] if w in vocab_set)
    total  = row['n_words']
    for j, w in enumerate(vocab):
        freq_matrix[i, j] = counts.get(w, 0) / total

# Z-score normalization per feature
mu    = freq_matrix.mean(axis=0)
sigma = freq_matrix.std(axis=0)
sigma[sigma == 0] = 1
z_matrix = (freq_matrix - mu) / sigma

# Pairwise Delta = mean(|z_i - z_j|) over all features
n = len(top_users)
delta_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = np.mean(np.abs(z_matrix[i] - z_matrix[j]))
        delta_matrix[i, j] = d
        delta_matrix[j, i] = d

delta_df_matrix = pd.DataFrame(
    delta_matrix,
    index=top_users['username'].tolist(),
    columns=top_users['username'].tolist(),
)
vals = delta_matrix[delta_matrix > 0]
print(f"\nMatriz Delta: {n}×{n} usuarios")
print(f"Delta medio: {vals.mean():.4f}  |  min: {vals.min():.4f}  |  max: {vals.max():.4f}")

In [ ]:
# Heatmap of Burrows' Delta — top-40 users
# Color inverted: dark/green = similar (low Delta) = potential sockpuppet
TOP_HEAT = 40
sub    = delta_df_matrix.iloc[:TOP_HEAT, :TOP_HEAT]
labels = sub.index.tolist()

fig = go.Figure(go.Heatmap(
    z=sub.values,
    x=labels, y=labels,
    colorscale='RdYlGn',
    reversescale=True,
    hovertemplate='%{y} vs %{x}<br>Delta=%{z:.4f}<extra></extra>',
    colorbar=dict(title='Delta<br>(↓ = más similar)'),
))
fig.update_layout(
    title="Burrows' Delta — top 40 usuarios IronMarch<br>"
          "<sup>Verde oscuro = estilo similar → candidato a misma persona</sup>",
    template='plotly_dark', height=720, width=760,
    xaxis=dict(tickangle=-45, tickfont=dict(size=8)),
    yaxis=dict(tickfont=dict(size=8)),
    margin=dict(l=140, r=20, t=80, b=140),
)
fig.show()

In [ ]:
# Sockpuppet candidates — pairs with lowest Delta
users_list = delta_df_matrix.index.tolist()
n = len(users_list)
pairs = []
name_to_nwords = dict(zip(delta_corpus['username'], delta_corpus['n_words']))

for i in range(n):
    for j in range(i + 1, n):
        pairs.append({
            'user_a':  users_list[i],
            'user_b':  users_list[j],
            'delta':   round(delta_df_matrix.iloc[i, j], 4),
            'words_a': name_to_nwords.get(users_list[i], 0),
            'words_b': name_to_nwords.get(users_list[j], 0),
        })

pairs_df = pd.DataFrame(pairs).sort_values('delta')
q01 = pairs_df['delta'].quantile(0.01)
q10 = pairs_df['delta'].quantile(0.10)

print("Pares con menor Delta (top 20):")
print(pairs_df.head(20).to_string(index=False))
print(f"\nPercentil  1%: Delta < {q01:.4f}  ({int(len(pairs_df)*0.01)} pares)")
print(f"Percentil 10%: Delta < {q10:.4f}  ({int(len(pairs_df)*0.10)} pares)")

suspicious = pairs_df[pairs_df['delta'] <= q01]
print(f"\nTop candidatos (percentil 1%, {len(suspicious)} pares):")
print(suspicious[['user_a', 'user_b', 'delta']].to_string(index=False))
print("""
\n⚠️  NOTA sobre usuario '255':
   username='255', userid=0 — patrón de superadmin en foros IPS.
   Casi con certeza Alexander Slavros, fundador de IronMarch.
   Con ~692k palabras domina el vocabulario del corpus; sus pares
   de Delta bajo deben interpretarse con cautela.""")

# Delta distribution
all_deltas = pairs_df['delta'].values
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_deltas, nbinsx=60, marker_color='steelblue', opacity=0.8))
fig.add_vline(x=q01, line_color='red',    line_dash='dash',
              annotation_text=f'P1% = {q01:.3f}', annotation_position='top right')
fig.add_vline(x=q10, line_color='orange', line_dash='dash',
              annotation_text=f'P10% = {q10:.3f}', annotation_position='top right')
fig.update_layout(
    title="Distribución de Burrows' Delta — todos los pares<br>"
          "<sup>Umbral de sospecha: percentil 1% (rojo) y 10% (naranja)</sup>",
    xaxis_title="Delta (↓ = más similar)",
    yaxis_title="Frecuencia",
    template='plotly_dark', height=420,
)
fig.show()

In [ ]:
# Temporal posting profiles per user — hourly UTC distribution
# High similarity = posting at same hours → same timezone / same person
date_col = 'dateline' if 'dateline' in posts_clean.columns else None

if date_col is None:
    print("Sin columna de fecha — omitiendo análisis temporal.")
else:
    posts_t = posts_clean.dropna(subset=[date_col]).copy()
    # Ensure datetime
    if not pd.api.types.is_datetime64_any_dtype(posts_t[date_col]):
        posts_t[date_col] = pd.to_datetime(posts_t[date_col], unit='s', errors='coerce')
    posts_t = posts_t.dropna(subset=[date_col])
    posts_t['hour'] = posts_t[date_col].dt.hour

    valid_users = set(delta_corpus['userid'].tolist())
    posts_t = posts_t[posts_t['userid'].isin(valid_users)]

    temporal_profiles: dict[int, np.ndarray] = {}
    for uid, group in posts_t.groupby('userid'):
        hist, _ = np.histogram(group['hour'], bins=24, range=(0, 24))
        if hist.sum() >= 20:
            temporal_profiles[uid] = hist / hist.sum()

    print(f"Usuarios con perfil temporal (≥20 posts): {len(temporal_profiles):,}")

    # Top-5 peak hours
    top5_uids = delta_corpus.head(5)['userid'].tolist()
    print("\nHora pico por usuario (top 5 más prolíficos):")
    for uid in top5_uids:
        if uid in temporal_profiles:
            peak  = int(np.argmax(temporal_profiles[uid]))
            uname = str(uid_to_name.get(uid, uid))
            print(f"  {uname:<25} pico a las {peak:02d}h UTC")

    # Cosine similarity between temporal histograms
    uids_t = list(temporal_profiles.keys())
    vecs_t = np.array([temporal_profiles[u] for u in uids_t])
    sim_t  = cosine_similarity(vecs_t)
    np.fill_diagonal(sim_t, 0)
    temporal_sim_df = pd.DataFrame(sim_t, index=uids_t, columns=uids_t)
    print(f"\nSimilitud temporal media: {sim_t[sim_t > 0].mean():.4f}")

In [ ]:
# Combined signal: low Delta + high temporal similarity → strong sockpuppet candidates
if 'temporal_sim_df' not in dir() or 'delta_df_matrix' not in dir():
    print("Ejecutar primero las celdas de Delta y temporal.")
else:
    common_uids = [
        u for u in temporal_sim_df.index
        if str(uid_to_name.get(u, u)) in delta_df_matrix.index
        and not pd.isna(uid_to_name.get(u, u))
    ]
    records = []
    for i in range(len(common_uids)):
        for j in range(i + 1, len(common_uids)):
            uid_a, uid_b = common_uids[i], common_uids[j]
            name_a = str(uid_to_name.get(uid_a, uid_a))
            name_b = str(uid_to_name.get(uid_b, uid_b))
            if name_a not in delta_df_matrix.index or name_b not in delta_df_matrix.index:
                continue
            records.append({
                'user_a':       name_a,
                'user_b':       name_b,
                'delta':        round(delta_df_matrix.loc[name_a, name_b], 4),
                'temporal_sim': round(temporal_sim_df.loc[uid_a, uid_b], 4),
            })

    combined = pd.DataFrame(records)
    if len(combined) == 0:
        print("Sin pares en común entre ambas matrices.")
    else:
        combined['delta_norm'] = 1 - (combined['delta'] - combined['delta'].min()) / (combined['delta'].max() - combined['delta'].min() + 1e-9)
        combined['t_norm']     = (combined['temporal_sim'] - combined['temporal_sim'].min()) / (combined['temporal_sim'].max() - combined['temporal_sim'].min() + 1e-9)
        combined['score']      = (combined['delta_norm'] + combined['t_norm']) / 2
        combined = combined.sort_values('score', ascending=False)

        print("Top 20 candidatos combinados (Delta bajo + horario similar):")
        print(combined[['user_a', 'user_b', 'delta', 'temporal_sim', 'score']].head(20).to_string(index=False))

        d_thresh = combined['delta'].quantile(0.10)
        t_thresh = combined['temporal_sim'].quantile(0.90)
        suspicious_combined = combined[(combined['delta'] <= d_thresh) & (combined['temporal_sim'] >= t_thresh)]

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=combined['delta'],
            y=combined['temporal_sim'],
            mode='markers',
            marker=dict(size=6, color=combined['score'], colorscale='YlOrRd',
                        reversescale=True, showscale=True,
                        colorbar=dict(title='Score<br>combinado')),
            text=combined['user_a'] + ' + ' + combined['user_b'],
            hovertemplate='<b>%{text}</b><br>Delta=%{x:.4f}<br>Temporal sim=%{y:.4f}<extra></extra>',
        ))
        if len(suspicious_combined) > 0:
            fig.add_trace(go.Scatter(
                x=suspicious_combined['delta'], y=suspicious_combined['temporal_sim'],
                mode='markers+text',
                marker=dict(size=12, color='red', symbol='star'),
                text=suspicious_combined['user_a'] + '+' + suspicious_combined['user_b'],
                textposition='top center', textfont=dict(size=8),
                name='sospechosos',
            ))
        fig.add_vline(x=d_thresh, line_dash='dash', line_color='orange',
                      annotation_text='P10 Delta', annotation_position='top left')
        fig.add_hline(y=t_thresh, line_dash='dash', line_color='orange',
                      annotation_text='P90 Temporal', annotation_position='top right')
        fig.update_layout(
            title="Señal combinada: Burrows' Delta vs similitud temporal<br>"
                  "<sup>Cuadrante inferior-izquierdo: estilo similar Y horario similar → candidatos a sockpuppet</sup>",
            xaxis_title="Burrows' Delta (↓ = más similar)",
            yaxis_title="Similitud temporal coseno (↑ = mismo horario)",
            template='plotly_dark', height=520,
        )
        fig.show()

        print(f"\nPares en cuadrante sospechoso (P10 Delta + P90 Temporal): {len(suspicious_combined)}")
        if len(suspicious_combined) > 0:
            print(suspicious_combined[['user_a', 'user_b', 'delta', 'temporal_sim']].to_string(index=False))

        print("""
── Hallazgo destacado ─────────────────────────────────────────────
Par: 255 (userid=0, ~Slavros) + Talleyrand (userid=16)
  Delta:              0.4637  (el más bajo del corpus)
  Similitud temporal: 0.9896  (prácticamente idéntico en horario)

  Talleyrand: email anicot@elon.edu (Elon University, NC)
              activo 2011-09-14 → 2014-02-26, luego desaparece
              pico a las 20h UTC | Slavros pico 22h UTC

  Interpretación: ambas señales apuntan en la misma dirección,
  pero la desaparición de Talleyrand en 2014 y el dominio del
  volumen de 255 introduce incertidumbre.
  Es el candidato más sólido — requiere análisis de contenido.
──────────────────────────────────────────────────────────────────""")


In [ ]:
# Stylometric similarity network — users connected if 1 - Delta/max_Delta >= threshold
# Use inverted-normalized Delta as similarity score
STYLO_THRESHOLD = 0.75

if 'delta_df_matrix' in dir() and len(delta_df_matrix) > 0:
    max_delta = delta_df_matrix.values[delta_df_matrix.values > 0].max()
    sim_matrix = 1 - (delta_df_matrix / max_delta)
    np.fill_diagonal(sim_matrix.values, 0)

    G_stylo = nx.Graph()
    users_list_delta = sim_matrix.index.tolist()
    for i in range(len(users_list_delta)):
        for j in range(i + 1, len(users_list_delta)):
            s = float(sim_matrix.iloc[i, j])
            if s >= STYLO_THRESHOLD:
                G_stylo.add_edge(users_list_delta[i], users_list_delta[j], weight=s)

    print(f"Umbral: {STYLO_THRESHOLD}  |  Nodos: {G_stylo.number_of_nodes()}  |  Aristas: {G_stylo.number_of_edges()}")

    if G_stylo.number_of_edges() > 0:
        pos = nx.spring_layout(G_stylo, k=2.0, iterations=60, seed=42)
        edge_x, edge_y = [], []
        for u, v in G_stylo.edges():
            x0, y0 = pos[u]; x1, y1 = pos[v]
            edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

        node_deg = [G_stylo.degree(n) for n in G_stylo.nodes()]
        fig = go.Figure(data=[
            go.Scatter(x=edge_x, y=edge_y, mode='lines',
                       line=dict(width=1.5, color='rgba(233,78,78,0.4)'), hoverinfo='none'),
            go.Scatter(
                x=[pos[n][0] for n in G_stylo.nodes()],
                y=[pos[n][1] for n in G_stylo.nodes()],
                mode='markers+text',
                marker=dict(size=[d * 8 + 10 for d in node_deg], color=node_deg,
                            colorscale='YlOrRd', showscale=True,
                            colorbar=dict(title='Conexiones')),
                text=list(G_stylo.nodes()),
                textposition='top center',
                textfont=dict(size=9, color='white'),
                hoverinfo='text',
            ),
        ])
        fig.update_layout(
            title=f'Red estilométrica IronMarch (similitud ≥ {STYLO_THRESHOLD})',
            template='plotly_dark', showlegend=False, width=900, height=620,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=20, r=20, t=60, b=20),
        )
        fig.show()
    else:
        print(f"Sin pares con similitud ≥ {STYLO_THRESHOLD}. Probar umbral menor.")
else:
    print("Ejecutar primero la celda de cálculo de Delta.")

## Sección 4 — Pivoting y atribución

Tabla resumen por usuario: volumen, estilo, topics (notebook 02), entidades NER (notebook 02),
subforo principal, horario activo.
Exportada como `user_profiles.parquet` para consumo en análisis posteriores.

In [ ]:
# User pivot table aggregating all available signals
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'
date_col = 'dateline' if 'dateline' in posts_clean.columns else None

# Ensure datetime
if date_col:
    if not pd.api.types.is_datetime64_any_dtype(posts_clean[date_col]):
        posts_clean[date_col] = pd.to_datetime(posts_clean[date_col], unit='s', errors='coerce')

# Base aggregation
agg: dict = {
    'post_count':    (text_col, 'count'),
    'avg_post_len':  (text_col, lambda x: x.dropna().astype(str).str.len().mean()),
}
if date_col:
    agg['first_post'] = (date_col, 'min')
    agg['last_post']  = (date_col, 'max')
    agg['active_days']= (date_col, lambda x: (x.max() - x.min()).days if pd.notna(x.max()) else 0)

user_pivot = posts_clean.groupby('userid').agg(**agg).reset_index()

# Username
user_pivot['username'] = user_pivot['userid'].map(uid_to_name)

# Primary forum (if forumid column available)
forum_col = 'forumid' if 'forumid' in posts_clean.columns else None
if forum_col:
    primary_forum = (
        posts_clean.groupby('userid')[forum_col]
        .agg(lambda x: x.value_counts().idxmax())
        .reset_index()
        .rename(columns={forum_col: 'primary_forum'})
    )
    user_pivot = user_pivot.merge(primary_forum, on='userid', how='left')

# Burrows' Delta — min Delta for each user (lower = more stylistically similar to someone)
if 'delta_df_matrix' in dir() and len(delta_df_matrix) > 0:
    name_to_min_delta = {}
    for uname in delta_df_matrix.index:
        row_vals = delta_df_matrix.loc[uname]
        row_vals = row_vals[row_vals > 0]
        name_to_min_delta[uname] = row_vals.min() if len(row_vals) > 0 else np.nan
    user_pivot['min_delta'] = user_pivot['username'].map(name_to_min_delta)

# Peak posting hour (UTC)
if 'temporal_profiles' in dir():
    user_pivot['peak_hour_utc'] = user_pivot['userid'].map(
        {uid: int(np.argmax(prof)) for uid, prof in temporal_profiles.items()}
    )

# NER: load from 02 results if available
ner_cache = IM_RESULTS / 'ner_comparison.parquet'
if ner_cache.exists():
    ner_all = pd.read_parquet(ner_cache)
    print(f"NER cargado: {len(ner_all):,} entidades")
    # Top entity type per user (if userid column present)
    if 'userid' in ner_all.columns and 'entity_type' in ner_all.columns:
        top_ner_type = (
            ner_all.groupby('userid')['entity_type']
            .agg(lambda x: x.value_counts().idxmax())
            .reset_index()
            .rename(columns={'entity_type': 'top_ner_type'})
        )
        user_pivot = user_pivot.merge(top_ner_type, on='userid', how='left')
else:
    print("NER cache no encontrado — omitiendo columna NER.")

user_pivot = user_pivot.sort_values('post_count', ascending=False).reset_index(drop=True)
print(f"\nUser profiles: {len(user_pivot):,} usuarios")
print(user_pivot.head(10).to_string())

out_path = IM_RESULTS / 'user_profiles.parquet'
user_pivot.to_parquet(out_path, index=False)
print(f"\nGuardado: {out_path}")

## Resumen ejecutivo

### Hallazgos principales

1. **Red completamente cohesionada** — el 100% de usuarios activos forma un único componente
   conectado. MOONLORD (Alexander Slavros) lidera el betweenness en la red pública;
   usuario `255` (userid=0, probable cuenta de admin) domina los mensajes privados
   con el 53% del betweenness. Dos capas de influencia distintas: pública y oculta.

2. **El muestreo de posts converge al full con rendimiento marginal decreciente** —
   top-50 (2h 35min) → ρ=0.799 | top-100 (3h 33min) → ρ=0.860 | top-150 (4h 12min) → ρ=0.896.
   El punto óptimo para foros de tamaño similar a IronMarch es **top-100**: 86% de fidelidad
   al embedding completo (~9h) con menos de 4 horas de cómputo.
   La diferencia metodológica sí importa: concatenar texto (A) vs. promediar embeddings (B/C/D/E)
   captura aspectos distintos del estilo.

3. **NER captura el ideario del foro** — 2,515 entidades en 1,500 posts.
   Top: Hitler (×28), Evola (×15), Golden Dawn (×10). La presencia de Evola
   casi al nivel de Hitler es un rasgo distintivo de IronMarch frente a otros
   foros de extrema derecha más populistas.

4. **Burrows' Delta + patrones temporales identifican candidatos a sockpuppet** —
   Delta medio del corpus = 1.08 (rango 0.46–2.03).
   Par más sospechoso: `255` + `Talleyrand` (Delta=0.46, similitud temporal=0.99).
   Talleyrand: `anicot@elon.edu`, activo solo 2011–2014 y luego desaparece.
   Requiere corroboración por contenido.

5. **Los atentados de París (nov 2015) son el único evento externo con spike estadístico
   confirmado** — 1.62× baseline, índice de activación = 2.35 σ.
   La reacción no es sobre supremacismo racial (`jew`, `white` bajan)
   sino islamofóbica: `paris` ×22, `refugee` ×16, `islam` ×4.
   El foro usa el atentado como munición narrativa anti-inmigración.

6. **Trump y Charlottesville no generan spike** — 1.03× y 0.96× baseline.
   Para 2016–2017 la comunidad ya estaba en declive activo,
   coherente con el cierre en noviembre de 2017.

### Por qué este dataset es excepcional

La existencia de **ground truth** — miembros identificados en investigaciones judiciales,
el fundador doxxeado, el servidor confiscado — permite validar el análisis computacional.

### Limitaciones y ética

- El análisis es exploratorio: ningún resultado es conclusivo sin corroboración independiente
- Los datos contienen información de personas reales; manejo mínimamente invasivo
- Objetivo defensivo: entender cómo operan estas redes, no publicar perfiles individuales
- Burrows' Delta con corpus homogéneo tiene menor poder discriminante; los candidatos son *leads*, no pruebas

In [ ]:
# Executive summary — programmatic summary of all computed results
print("=" * 60)
print("CASO 3 — IRONMARCH — RESUMEN EJECUTIVO")
print("=" * 60)

print(f"\nDATASET")
print(f"  Usuarios:  {len(users_clean):,}")
print(f"  Posts:     {len(posts_clean):,}")

print(f"\nEMBEDDINGS")
for label, pattern in [
    ('A — embed_users',         's5a_embed_users_*.npz'),
    ('B — centroids_full',      's5b_centroids_full_*.npz'),
    ('C — centroids_sampled50', 's5c_centroids_sampled_*.npz'),
    ('D — centroids_sampled100','s5d_centroids_sampled100_*.npz'),
    ('E — centroids_sampled150','s5e_centroids_sampled150_*.npz'),
]:
    m = sorted(IM_RESULTS.glob(pattern))
    status = m[-1].name if m else 'no generado'
    print(f"  {label}: {status}")

if spearman_results:
    print("  Spearman vs B:")
    for lbl, rho in spearman_results.items():
        print(f"    {lbl}: ρ={rho:.4f}")

print(f"\nESTILOMETRIA — BURROWS' DELTA")
if 'delta_df_matrix' in dir() and len(delta_df_matrix) > 0:
    _v = delta_df_matrix.values
    _m = _v > 0
    print(f"  Usuarios analizados: {len(delta_df_matrix):,}")
    print(f"  Delta medio: {_v[_m].mean():.4f}  |  min: {_v[_m].min():.4f}  |  max: {_v[_m].max():.4f}")
    if 'combined' in dir() and len(combined) > 0:
        top1 = combined.iloc[0]
        print(f"  Top candidato: {top1['user_a']} + {top1['user_b']} "
              f"(Delta={top1['delta']:.4f}, temporal={top1['temporal_sim']:.4f})")
else:
    print("  Delta no calculado")

print(f"\nPERFILES DE USUARIO")
if (IM_RESULTS / 'user_profiles.parquet').exists():
    _up = pd.read_parquet(IM_RESULTS / 'user_profiles.parquet')
    print(f"  Guardados: {len(_up):,} perfiles en user_profiles.parquet")
else:
    print("  user_profiles.parquet no generado")

print("=" * 60)